In [ ]:
# importing the libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
# setting up the paths and importing the dataset
DATA_DIR = Path("Data download/data")
INPUT_PATH = DATA_DIR / "filings_clean.csv"
OUTPUT_PATH = DATA_DIR / "filings_gpt.csv"
CHECKPOINT_PATH = DATA_DIR / "gpt_checkpoint.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings")
print(f"Downside rate: {df['downside'].mean():.1%}")

In [ ]:
# setting up the OpenAI client
# API key is loaded from a .env file (NEVER commit .env to git)
# See .env.example for the required format
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Copy .env.example to .env "
        "and add your key, or set the environment variable."
    )

client = OpenAI(api_key=api_key)

# Loading the GPT-4o system prompt from the standalone prompt file
# This is the same prompt reproduced in Appendix A of the thesis
PROMPT_PATH = Path("../prompts/gpt4o_system_prompt.txt")
PROMPT = PROMPT_PATH.read_text(encoding="utf-8")
print(f"Loaded prompt: {len(PROMPT)} characters")

In [ ]:
# defining the function to score the filings using the LLM
def score_gpt(text: str, max_chars: int = 20000) -> dict:
    ''''Scores the given text using the GPT model and returns a dictionary with the score and any error message.'''
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {'gpt_score': np.nan, 'gpt_error': 'empty'}
    # Take first 14000 + last 6000 chars to capture both
    # financial results (front) and guidance/outlook (back)
    if len(text) > max_chars:
        front = text[:14000]
        back  = text[-6000:]
        text  = front + "\n[...]\n" + back

    try:
        response = client.chat.completions.create(
            model="gpt-4o-2024-05-13",  # pin exact version for reproducibility
            messages=[
                {"role": "system", "content": PROMPT},
                {"role": "user",   "content": text}
            ],
            max_tokens=10,
            temperature=0
        )
        raw   = response.choices[0].message.content.strip()
        score = float(re.search(r'-?\d+\.?\d*', raw).group())
        score = max(-1.0, min(1.0, score))
        return {'gpt_score': score, 'gpt_error': None}

    except Exception as e:
        return {'gpt_score': np.nan, 'gpt_error': str(e)}

In [ ]:
CHECKPOINT_EVERY = 10 

# clearing old checkpoint to ensure clean run with updated prompt
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("Old checkpoint cleared.")

# using stratified sampling to create a balanced dataset of 2000 filings (1000 downside, 1000 non-downside) for scoring with the LLM
downside_sample    = df[df['downside']==1].sample(n=1000, random_state=42)
nondownside_sample = df[df['downside']==0].sample(n=1000, random_state=42)
target_df = pd.concat([downside_sample, nondownside_sample]).sample(frac=1, random_state=42)

print(f"Stratified sample : {len(target_df)} filings")
print(f"  Downside        : {target_df['downside'].sum()}")
print(f"  Non-downside    : {(target_df['downside']==0).sum()}")

results = []
for i, (idx, row) in enumerate(tqdm(target_df.iterrows(), total=len(target_df), desc="GPT-4o scoring")):
    result = score_gpt(row['cleanText'])
    result['accessionNumber'] = row['accessionNumber']
    result['ticker']          = row['ticker']
    result['filingDate']      = row['filingDate']
    result['car_0_1']         = row['car_0_1']
    result['downside']        = row['downside']
    results.append(result)
    time.sleep(0.5)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
        tqdm.write(f"Checkpoint saved at {len(results)} filings")

# saving the final results
pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
print(f"Scoring complete. {len(results)} filings saved.")

In [ ]:
# Load scored filings
scores_df = pd.read_csv(CHECKPOINT_PATH)

# Retrospective 80/20 split, stratified to preserve 50/50 balance
val_df, test_df = train_test_split(
    scores_df,
    test_size=0.80,
    stratify=scores_df['downside'],
    random_state=42
)

print(f"Validation set : {len(val_df)} filings")
print(f"  Downside     : {val_df['downside'].sum()}")
print(f"Test set       : {len(test_df)} filings")
print(f"  Downside     : {test_df['downside'].sum()}")

# Report AUC on test set only
auc_test = roc_auc_score(test_df['downside'], -test_df['gpt_score'])
print(f"\nGPT-4o AUC (test set, n=1600) : {auc_test:.4f}")

# Bootstrap CI on test set
np.random.seed(42)
boot_aucs = []
for _ in range(1000):
    sample = test_df.sample(len(test_df), replace=True)
    try:
        boot_aucs.append(roc_auc_score(sample['downside'], -sample['gpt_score']))
    except:
        pass

ci_low  = np.percentile(boot_aucs, 2.5)
ci_high = np.percentile(boot_aucs, 97.5)
print(f"95% bootstrap CI              : [{ci_low:.4f}, {ci_high:.4f}]")

# Save both splits for later use with FinBERT and LM dictionary comparison
val_df.to_csv(DATA_DIR / "gpt_val.csv", index=False)
test_df.to_csv(DATA_DIR / "gpt_test.csv", index=False)
print(f"\nSaved val and test splits to {DATA_DIR}")

In [ ]:
# Force exactly 10% in each tail regardless of ties
pilot_df_sorted = pilot_df.sort_values('gpt_score')
n = len(pilot_df_sorted)
bottom_40 = pilot_df_sorted.head(int(n * 0.10))
top_40    = pilot_df_sorted.tail(int(n * 0.10))

print(f"Bottom 10% avg CAR : {bottom_40['car_0_1'].mean():.4f}  (n={len(bottom_40)})")
print(f"Top 10% avg CAR    : {top_40['car_0_1'].mean():.4f}  (n={len(top_40)})")
print(f"Spread             : {bottom_40['car_0_1'].mean() - top_40['car_0_1'].mean():.4f}")